# Exploratory Analysis of Public Bus Stop Accessibility and Service Gaps Across Chennai

This notebook analyzes the supplied Chennai bus stop audit dataset. It focuses on pedestrian accessibility, boarding conditions, encroachment, safety, shelter quality, lighting, route information and service gaps.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
df = pd.read_csv('../Dataset/dataset.csv')
print('Dataset shape:', df.shape)
df.head()

## 1. Data Quality and Missing Values

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing>0].head(20)

## 2. Data Cleaning

In [ ]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].map(lambda x: x.strip() if isinstance(x,str) else x)
df['Date and Time'] = pd.to_datetime(df['Date and Time'], errors='coerce')
df = df.drop_duplicates().reset_index(drop=True)
access_cols = [
    'Is there space behind the bus shelter for pedestrians to walk/wheelchair to move freely?',
    'Is there space between the bus shelter and the road for pedestrians to walk/wheelchair to move freely?\u00a0',
    'Can pedestrians/wheelchair users move past the bus stop on the footpath, without having to step down on the road?\u00a0'
]
df['pedestrian_access_score'] = df[access_cols].eq('Yes').sum(axis=1)
enc_map={'Yes – fully accessible, no obstructions':2,'Partially encroached – e.g., vendors, parked two-wheelers, auto-rickshaws, construction material':1,'Heavily encroached – the stop is largely unusable because of obstructions':0}
df['encroachment_score']=df['Is the bus stop area free of encroachments?'].map(enc_map)
df['boarding_access_good']=df['How far away from the kerb/Bus stop platform does the bus stop?'].eq('Perfect / Flush (Less than 30 cm): Passengers can step directly from the footpath onto the bus.')
print(df.shape)

## 3. Exploratory Analysis

In [ ]:
print('Pedestrian access score distribution')
print(df['pedestrian_access_score'].value_counts().sort_index())
print('\nShelter condition')
print(df['Condition of Bus Shelter'].value_counts(dropna=False))

## 4. Accessibility Findings

In [ ]:
indicators = {
    'Behind shelter space': (df[access_cols[0]]=='Yes').mean()*100,
    'Road-side space': (df[access_cols[1]]=='Yes').mean()*100,
    'Footpath continuity': (df[access_cols[2]]=='Yes').mean()*100,
    'Flush/good boarding': df['boarding_access_good'].mean()*100,
    'Fully clear of encroachment': (df['encroachment_score']==2).mean()*100
}
pd.Series(indicators).round(1)

## 5. Visualizations

In [ ]:
df['Condition of Bus Shelter'].fillna('Missing').value_counts().plot(kind='bar', figsize=(9,5), title='Bus Shelter Condition Distribution')
plt.ylabel('Number of bus stops'); plt.xticks(rotation=35, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
df['Is the bus stop area free of encroachments?'].value_counts().plot(kind='bar', figsize=(10,5), title='Encroachment Levels')
plt.ylabel('Number of bus stops'); plt.xticks(rotation=35, ha='right'); plt.tight_layout(); plt.show()

## 6. Conclusion

The audit indicates substantial pedestrian-access constraints: many stops lack usable movement space around shelters, and only a minority have flush boarding conditions. Encroachment, lighting, route-information availability and shelter capacity are key areas for improvement. These results can guide targeted infrastructure and service-information upgrades.